# تدريب نواة 40 مليون وزن على Colab (GPU)
قبل البدء: **Runtime ← Change runtime type ← T4 GPU**.
الحجم وحده لا يفهّم: كل ما أضفت **بيانات** (نص عربي وبرمجي نظيف) صار أذكى. هذا الدفتر يوسّع النموذج الحالي إلى 40 مليون بحيث يبدأ مطابقا للقديم، ثم يدرّبه.

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'لا يوجد! فعّل GPU')

**1) ارفع `Nuwaa.zip`** (آخر نسخة أعطيتك). اختياريا: ملف نصي `extra.txt` فيه نصوصك (سطر لكل نص).

In [ ]:
from google.colab import files
up = files.upload()
name = [k for k in up if k.endswith('.zip')][0]
!rm -rf Nuwaa && mkdir Nuwaa && unzip -q "$name" -d Nuwaa
%cd Nuwaa
!python -m nuwaa.lm.hw

**2) وسّع إلى 40 مليون** (يبدأ مطابقا للنموذج الحالي).

In [ ]:
!python tools/scale_up.py --target 40000000 --out scale/nlm_40m_init.npz

**3) درّب.** `STEPS` الأكبر = وقت أطول وذكاء أكثر. ابدأ بـ 3000 لترى الوقت. الفحص الذاتي في البداية يجب أن يطبع `self-check` بفرق أقل من 0.03 وإلا توقف.

**3) درّب في الخلفية** (ما يوقف الدفتر). تقدر تكمل تشغيل خلايا ثانية وأنت تتدرب. كل نسخة أفضل تنحفظ تلقائيا.

In [ ]:
import os, subprocess
STEPS = 20000
EXTRA = '--text extra.txt' if os.path.exists('extra.txt') else ''
subprocess.Popen(f'python tools/train_gru_torch.py --init scale/nlm_40m_init.npz --out weights/nlm_40m.npz --steps {STEPS} --bs 64 --amp {EXTRA} > train.log 2>&1', shell=True)
print('بدأ التدريب في الخلفية. آخر سطور السجل:')

In [ ]:
!tail -n 5 train.log

**4) كلمه وهو يتدرب.** كل مرة تشغّل هذي الخلية تحمّل آخر نسخة محفوظة وتكلمه بها. اكتب في `QUESTION` وشغّل.

In [ ]:
import os, sys
sys.path.insert(0, 'Nuwaa' if os.path.exists('Nuwaa') else '.')
from nuwaa.lm.gru import GRULM, Session
QUESTION = 'ما هي بايثون؟'
if os.path.exists('weights/nlm_40m.npz'):
    m = GRULM.load('weights/nlm_40m.npz')
    print(Session(m, seed=1).reply(QUESTION, learn=False, max_chars=400, temp=0.6)[0])
else:
    print('ما فيه نسخة محفوظة بعد، انتظر دقيقة وشغّل الخلية مرة ثانية.')

**5) لما يخلص التدريب** (تشوف `done` في `train.log`)، حوّله لصيغة الجوال:

In [ ]:
from nuwaa.lm.gru import GRULM
open('nlm2.bin','wb').write(GRULM.load('weights/nlm_40m.npz').to_bytes())
files.download('nlm2.bin')